# Isla-SNN v3 Training 🧠⚡

Spiking Neural Network language model with Spike Synchrony Attention.

**v3 improvements:**
- 🔥 Gated additive residual (spikes always active + learnable boost)
- 🔥 8 timesteps (256 temporal patterns per neuron)
- 🧬 Membrane potential as continuous feature (spike + α·membrane)
- 🧬 Spike Frequency Adaptation (dynamic threshold)

**GPU**: L4 (24GB) or 4060 Ti (16GB) with gradient checkpointing.

In [ ]:
!pip install -q torch transformers datasets wandb gdown

In [ ]:
!git clone https://github.com/Heitorkk2/Isla-SNN.git 2>/dev/null || echo 'Already cloned'
%cd Isla-SNN

In [ ]:
import isla
import torch
print(f"isla v{isla.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Dataset

Option A: Download pre-tokenized dataset from Google Drive.  
Option B: Use your own JSONL file (Isla tokenizes automatically).

In [ ]:
import os

# ─── Option A: Google Drive (pre-tokenized) ───
# import gdown, zipfile
# GDRIVE_FILE_ID = "YOUR_FILE_ID_HERE"
# OUTPUT_ZIP = "/root/isla_dataset.zip"
# DATASET_PATH = "/root/isla_dataset"
# if not os.path.exists(DATASET_PATH):
#     gdown.download(f"https://drive.google.com/uc?id={GDRIVE_FILE_ID}", OUTPUT_ZIP)
#     import zipfile; zipfile.ZipFile(OUTPUT_ZIP).extractall(DATASET_PATH)

# ─── Option B: Local JSONL (auto-tokenized) ───
DATASET_PATH = "./data/corpus_v3.jsonl"

print(f"Dataset: {DATASET_PATH}")
if os.path.isfile(DATASET_PATH):
    size_gb = os.path.getsize(DATASET_PATH) / 1e9
    print(f"Size: {size_gb:.1f} GB")

## Configuration

v3 architecture: **~150M params** (768d × 12L × 12H × 8T)

In [ ]:
model_config = isla.ModelConfig(
    hidden_dim=768,
    num_layers=12,
    num_heads=12,
    num_timesteps=8,
    max_seq_len=2048,
    dropout=0.1,
    beta_init=0.5,
    threshold=0.3,
    surrogate_slope=25.0,
    sync_tau_init=1.0,
    spike_reg_lambda=0.05,
    target_spike_rate=0.3,
    tokenizer_name="NousResearch/Llama-2-7b-hf",
)

train_config = isla.TrainConfig(
    lr=3e-4,
    max_steps=23000,
    warmup_steps=1000,
    batch_size=4,
    gradient_accumulation_steps=8,
    bf16=True,
    gradient_checkpointing=True,
    log_every=50,
    eval_every=500,
    wandb=isla.WandbConfig(
        enabled=True,
        project="isla-snn-v3",
        run_name="v3-150m-gated-8ts-sfa-trilingual",
    ),
    checkpoint=isla.CheckpointConfig(
        output_dir="./outputs/v3_150m",
        save_every=2000,
        # resume_from="./outputs/v3_150m/latest",  # uncomment to resume
    ),
)

data_config = isla.DataConfig(
    dataset_path=DATASET_PATH,
    tokenizer_name="NousResearch/Llama-2-7b-hf",
    max_seq_len=2048,
    pack_sequences=True,
)

print("Configs ready.")
print(f"Effective batch size: {train_config.effective_batch_size}")
print(f"Tokens per step: {train_config.effective_batch_size * 2048:,}")
print(f"Total tokens: {train_config.effective_batch_size * 2048 * 23000:,} (~1.5B)")

## Train

In [ ]:
model, tokenizer = isla.train(model_config, train_config, data_config)

## Generate Text

In [ ]:
import isla
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading model on: {device}")

model, tokenizer = isla.load_model("./outputs/v3_150m/latest", device=device)
model.eval()

prompts = [
    "The future of spiking neural networks",
    "def fibonacci(n):",
    "A inteligência artificial",
]

for prompt in prompts:
    print(f"\nPrompt: {prompt}\n---")
    with torch.no_grad():
        for word in isla.generate_stream(
            model, tokenizer, prompt,
            max_new_tokens=80,
            temperature=0.8,
            device=device
        ):
            print(word, end="", flush=True)
    print("\n")

## Load from Checkpoint

On a fresh session:

In [ ]:
# model, tokenizer = isla.load_model("./outputs/v3_150m/final", device="cuda")
# print(isla.generate(model, tokenizer, "Hello world", device="cuda"))